# Data preview + ML feasibility (futures return as response)

Goal: inspect panel quality and try baseline models where `futures_return` is the response and all other variables are features.

In [4]:
import numpy as np
import pandas as pd
import warnings
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import HistGradientBoostingRegressor
from IPython.display import display

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tools.sm_exceptions import ConvergenceWarning as SMConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning as SMValueWarning

# Optional booster models. If package is unavailable, notebook continues gracefully.
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data/processed/monthly_panel.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run scripts/monthly_pipeline.ipynb first to generate data/processed/monthly_panel.csv")

panel = pd.read_csv(DATA_PATH)
panel["date"] = pd.to_datetime(panel["date"])
panel = panel.sort_values(["commodity", "date"]).reset_index(drop=True)

print(panel.shape)
print('xgboost available:', HAS_XGB, '| lightgbm available:', HAS_LGBM)
panel.head()

# Silence noisy infra warnings in notebook runs
warnings.filterwarnings("ignore", message="Could not find the number of physical cores")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", category=SMConvergenceWarning)
warnings.filterwarnings("ignore", category=SMValueWarning)
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")
warnings.filterwarnings("ignore", message="'M' is deprecated and will be removed in a future version")


(1446, 24)
xgboost available: True | lightgbm available: True


In [5]:
print("Columns:", panel.columns.tolist())
print("\nMissing values:")
print(panel.isna().mean().sort_values(ascending=False).head(15))

print("\nDate range:", panel["date"].min().date(), "->", panel["date"].max().date())
print("Commodities:", panel["commodity"].unique())


Columns: ['date', 'price_proxy', 'commodity', 'price_source', 'price_source_detail', 'price_series_desc', 'price_unit', 'futures_price', 'futures_return', 'usd_index', 'usd_index_source', 'interest_rate', 'vix', 'vix_source', 'temperature_anomaly', 'precipitation_anomaly', 'drought_index', 'extreme_heat_events', 'enso_oni', 'disaster_event_count', 'disaster_storm_count', 'disaster_fire_count', 'disaster_flood_count', 'climate_location_code']

Missing values:
date                     0.0
price_proxy              0.0
disaster_flood_count     0.0
disaster_fire_count      0.0
disaster_storm_count     0.0
disaster_event_count     0.0
enso_oni                 0.0
extreme_heat_events      0.0
drought_index            0.0
precipitation_anomaly    0.0
temperature_anomaly      0.0
vix_source               0.0
vix                      0.0
interest_rate            0.0
usd_index_source         0.0
dtype: float64

Date range: 1986-01-31 -> 2026-02-28
Commodities: ['corn' 'soybean' 'wheat']


In [6]:
# More robust feature engineering for low-signal monthly return prediction
model_df = panel.copy()

# Predict next-month return, so every predictor is based on information available at t.
model_df["target_next_return"] = model_df.groupby("commodity")["futures_return"].shift(-1)
model_df["month_num"] = model_df["date"].dt.month
model_df["month"] = model_df["date"].dt.month.astype(str)
model_df["time_idx"] = (model_df["date"].dt.year - model_df["date"].dt.year.min()) * 12 + model_df["date"].dt.month

base_lag_cols = [
    "futures_return",
    "futures_price",
    "usd_index",
    "interest_rate",
    "vix",
    "enso_oni",
    "disaster_event_count",
    "disaster_storm_count",
    "disaster_fire_count",
    "disaster_flood_count",
]
base_lag_cols = [c for c in base_lag_cols if c in model_df.columns]

for col in base_lag_cols:
    g = model_df.groupby("commodity")[col]
    for lag in [1, 2, 3, 6, 12]:
        model_df[f"{col}_lag{lag}"] = g.shift(lag)

# Simple momentum / volatility features from lagged return history
ret_group = model_df.groupby("commodity")["futures_return"]
for win in [3, 6, 12]:
    model_df[f"ret_mean_{win}"] = ret_group.transform(lambda s: s.shift(1).rolling(win).mean())
    model_df[f"ret_std_{win}"] = ret_group.transform(lambda s: s.shift(1).rolling(win).std())

target_col = "target_next_return"
model_df = model_df.dropna(subset=[target_col]).copy()

missing_rate = model_df.isna().mean()
sparse_cols = [c for c in model_df.columns if c not in [target_col, "date"] and missing_rate[c] > 0.90]
if sparse_cols:
    print("Dropping sparse columns:", sparse_cols)
    model_df = model_df.drop(columns=sparse_cols)

feature_cols = [c for c in model_df.columns if c not in [target_col, "date"]]


class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower_q=0.01, upper_q=0.99):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        x = np.asarray(X, dtype=float)
        self.lo_ = np.nanquantile(x, self.lower_q, axis=0)
        self.hi_ = np.nanquantile(x, self.upper_q, axis=0)
        return self

    def transform(self, X):
        x = np.asarray(X, dtype=float)
        return np.clip(x, self.lo_, self.hi_)


models = {
    "seasonal_naive": None,
    "naive_lag1": None,
    "ridge": Ridge(alpha=3.0),
    "elastic_net": ElasticNet(alpha=0.003, l1_ratio=0.25, random_state=42, max_iter=10000),
    "hist_gbdt": HistGradientBoostingRegressor(
        max_depth=3,
        learning_rate=0.03,
        max_iter=250,
        random_state=42,
    ),
}

# Booster models: add only if dependency exists.
if HAS_XGB:
    models["xgboost"] = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=1,
    )
if HAS_LGBM:
    models["lightgbm"] = LGBMRegressor(
        n_estimators=300,
        max_depth=-1,
        num_leaves=31,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=1,
        force_col_wise=True,
        verbose=-1,
    )

def build_target_variant(train_df, test_df, target_col, variant):
    y_train_raw = train_df[target_col].copy()
    y_test_raw = test_df[target_col].copy()

    if variant == "raw":
        return y_train_raw, y_test_raw, y_test_raw, {"test_seasonal": np.zeros(len(test_df))}

    if variant not in ["deseason", "deseason_winsor"]:
        raise ValueError(f"Unknown target variant: {variant}")

    seasonal_map = (
        train_df
        .assign(_y=y_train_raw)
        .groupby(["commodity", "month_num"])["_y"]
        .mean()
        .to_dict()
    )
    global_mean = float(y_train_raw.mean())

    train_seasonal = train_df.apply(
        lambda r: seasonal_map.get((r["commodity"], r["month_num"]), global_mean),
        axis=1,
    )
    test_seasonal = test_df.apply(
        lambda r: seasonal_map.get((r["commodity"], r["month_num"]), global_mean),
        axis=1,
    )

    y_train = y_train_raw - train_seasonal.to_numpy()
    y_test = y_test_raw - test_seasonal.to_numpy()

    if variant == "deseason_winsor":
        clip_lo = float(np.quantile(y_train, 0.01))
        clip_hi = float(np.quantile(y_train, 0.99))
        y_train = y_train.clip(clip_lo, clip_hi)
        y_test = y_test.clip(clip_lo, clip_hi)

    return y_train, y_test, y_test_raw, {"test_seasonal": test_seasonal.to_numpy()}

def evaluate_window(train_df, test_df, feature_cols, target_col, models, window_name, target_variant):
    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    y_train, y_test_transformed, y_test_raw, target_meta = build_target_variant(
        train_df=train_df,
        test_df=test_df,
        target_col=target_col,
        variant=target_variant,
    )

    numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]

    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("winsor", Winsorizer(lower_q=0.01, upper_q=0.99)),
                ("scaler", RobustScaler()),
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), categorical_cols),
        ]
    )

    seasonal_baseline = train_df.groupby(["commodity", "month_num"])[target_col].mean().to_dict()
    seasonal_global = float(train_df[target_col].mean())

    rows = []
    for name, estimator in models.items():
        if name == "seasonal_naive":
            pred_raw = test_df.apply(
                lambda r: seasonal_baseline.get((r["commodity"], r["month_num"]), seasonal_global),
                axis=1,
            ).to_numpy()
        elif name == "naive_lag1":
            naive_col = "futures_return_lag1"
            if naive_col not in X_test.columns:
                continue
            pred_raw = X_test[naive_col].fillna(float(y_train.mean())).to_numpy()
        else:
            pipe = Pipeline([
                ("preprocess", preprocess),
                ("model", estimator),
            ])
            pipe.fit(X_train, y_train)
            pred_transformed = pipe.predict(X_test)

            if target_variant == "raw":
                pred_raw = pred_transformed
            else:
                pred_raw = pred_transformed + target_meta["test_seasonal"]

        rmse = mean_squared_error(y_test_raw, pred_raw) ** 0.5
        rows.append({
            "window": window_name,
            "target_variant": target_variant,
            "model": name,
            "mae": mean_absolute_error(y_test_raw, pred_raw),
            "rmse": rmse,
            "r2": r2_score(y_test_raw, pred_raw),
            "direction_acc": (np.sign(pred_raw) == np.sign(y_test_raw)).mean(),
            "n_test": len(test_df),
            "date_start": test_df["date"].min(),
            "date_end": test_df["date"].max(),
        })

    return pd.DataFrame(rows)

# Quick noise diagnostic: how much variance drops after de-seasonalization + winsorization
seasonal_full = model_df.groupby(["commodity", "month_num"])[target_col].transform("mean")
target_deseason = model_df[target_col] - seasonal_full
d_lo = target_deseason.quantile(0.01)
d_hi = target_deseason.quantile(0.99)
target_deseason_winsor = target_deseason.clip(d_lo, d_hi)

noise_diag = pd.DataFrame({
    "series": ["raw_target", "deseason_target", "deseason_winsor_target"],
    "std": [model_df[target_col].std(), target_deseason.std(), target_deseason_winsor.std()],
    "iqr": [
        model_df[target_col].quantile(0.75) - model_df[target_col].quantile(0.25),
        target_deseason.quantile(0.75) - target_deseason.quantile(0.25),
        target_deseason_winsor.quantile(0.75) - target_deseason_winsor.quantile(0.25),
    ],
})
print("Target noise diagnostic:")
display(noise_diag)

# 1) Final holdout: last 24 months
unique_months = np.sort(model_df["date"].unique())
if len(unique_months) < 48:
    raise ValueError("Need at least 48 monthly points for holdout + rolling validation")

final_cutoff = unique_months[-24]
train_holdout = model_df[model_df["date"] < final_cutoff].copy()
test_holdout = model_df[model_df["date"] >= final_cutoff].copy()

target_variants = ["raw", "deseason_winsor"]
rolling_model_names = ["seasonal_naive", "elastic_net", "hist_gbdt", "xgboost", "lightgbm"]
models_for_rolling = {k: models[k] for k in rolling_model_names if k in models}
holdout_frames = []
for target_variant in target_variants:
    holdout_frames.append(
        evaluate_window(
            train_df=train_holdout,
            test_df=test_holdout,
            feature_cols=feature_cols,
            target_col=target_col,
            models=models,
            window_name="final_24m",
            target_variant=target_variant,
        )
    )

holdout_results = pd.concat(holdout_frames, ignore_index=True).sort_values(["target_variant", "rmse"])

print("Final holdout results (last 24 months):")
display(holdout_results)

# 2) Rolling-origin validation (fast mode for iteration)
rolling_results = []
quick_mode = True
if quick_mode:
    window_len = 24
    step = 24
    start_test_idx = 144  # faster iteration, still keeps 12y initial train
else:
    window_len = 24
    step = 12
    start_test_idx = 120

for start in range(start_test_idx, len(unique_months) - window_len + 1, step):
    test_start = unique_months[start]
    test_end = unique_months[start + window_len - 1]

    train_df = model_df[model_df["date"] < test_start].copy()
    test_df = model_df[(model_df["date"] >= test_start) & (model_df["date"] <= test_end)].copy()

    if train_df.empty or test_df.empty:
        continue

    wname = f"{pd.Timestamp(test_start).date()}__{pd.Timestamp(test_end).date()}"
    for target_variant in target_variants:
        rolling_results.append(
            evaluate_window(
                train_df=train_df,
                test_df=test_df,
                feature_cols=feature_cols,
                target_col=target_col,
                models=models_for_rolling,
                window_name=wname,
                target_variant=target_variant,
            )
        )

if rolling_results:
    rolling_results_df = pd.concat(rolling_results, ignore_index=True)
    rolling_summary = (
        rolling_results_df
        .groupby(["target_variant", "model"], as_index=False)
        .agg(
            mae_mean=("mae", "mean"),
            rmse_mean=("rmse", "mean"),
            rmse_median=("rmse", "median"),
            r2_mean=("r2", "mean"),
            r2_median=("r2", "median"),
            direction_acc_mean=("direction_acc", "mean"),
            n_windows=("window", "nunique"),
        )
        .sort_values(["target_variant", "rmse_mean"])
    )

    print("Rolling-origin summary:")
    display(rolling_summary)
else:
    print("No rolling windows were generated.")

# Persist key summaries for quick comparison across notebook runs
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
holdout_results.to_csv(OUTPUT_DIR / "model_holdout_results.csv", index=False)
if "rolling_summary" in locals():
    rolling_summary.to_csv(OUTPUT_DIR / "model_rolling_summary.csv", index=False)

Target noise diagnostic:


,series,std,iqr
0,raw_target,0.049444,0.051023
1,deseason_target,0.045432,0.047424
2,deseason_winsor_target,0.043438,0.047424


Final holdout results (last 24 months):


,window,target_variant,model,mae,rmse,r2,direction_acc,n_test,date_start,date_end
7,final_24m,deseason_winsor,seasonal_naive,0.020976,0.027549,0.301403,0.652778,72,2024-02-29,2026-01-31
12,final_24m,deseason_winsor,xgboost,0.021758,0.028831,0.234857,0.611111,72,2024-02-29,2026-01-31
10,final_24m,deseason_winsor,elastic_net,0.021842,0.028872,0.232697,0.611111,72,2024-02-29,2026-01-31
9,final_24m,deseason_winsor,ridge,0.023012,0.029681,0.189093,0.597222,72,2024-02-29,2026-01-31
11,final_24m,deseason_winsor,hist_gbdt,0.023502,0.029803,0.182402,0.611111,72,2024-02-29,2026-01-31
13,final_24m,deseason_winsor,lightgbm,0.023210,0.029809,0.182083,0.541667,72,2024-02-29,2026-01-31
8,final_24m,deseason_winsor,naive_lag1,0.037634,0.047731,-1.097155,0.472222,72,2024-02-29,2026-01-31
0,final_24m,raw,seasonal_naive,0.020976,0.027549,0.301403,0.652778,72,2024-02-29,2026-01-31
4,final_24m,raw,hist_gbdt,0.023988,0.030179,0.161615,0.583333,72,2024-02-29,2026-01-31
5,final_24m,raw,xgboost,0.024420,0.030467,0.145557,0.583333,72,2024-02-29,2026-01-31


Rolling-origin summary:


,target_variant,model,mae_mean,rmse_mean,rmse_median,r2_mean,r2_median,direction_acc_mean,n_windows
4,deseason_winsor,xgboost,0.034445,0.045662,0.045677,0.059071,0.031686,0.608135,14
0,deseason_winsor,elastic_net,0.034718,0.045806,0.045581,0.051361,0.110825,0.615079,14
1,deseason_winsor,hist_gbdt,0.034552,0.045873,0.045955,0.055454,0.071793,0.611111,14
2,deseason_winsor,lightgbm,0.035398,0.046561,0.044513,0.021854,-0.037497,0.606151,14
3,deseason_winsor,seasonal_naive,0.034804,0.046604,0.048286,0.030184,0.025183,0.590278,14
9,raw,xgboost,0.034130,0.044830,0.044717,0.093437,0.127846,0.625000,14
7,raw,lightgbm,0.034495,0.045322,0.044251,0.067947,0.038983,0.616071,14
6,raw,hist_gbdt,0.034399,0.045451,0.045024,0.069348,0.101421,0.614087,14
8,raw,seasonal_naive,0.034804,0.046604,0.048286,0.030184,0.025183,0.590278,14
5,raw,elastic_net,0.035452,0.046676,0.045633,0.018216,0.082083,0.591270,14


In [7]:
# Commodity-wise rolling validation (same settings) to see where signal really exists
per_commodity_results = []

for commodity_name, sub in model_df.groupby("commodity"):
    sub = sub.sort_values("date").copy()
    sub_months = np.sort(sub["date"].unique())

    if len(sub_months) < (start_test_idx + window_len):
        continue

    for start in range(start_test_idx, len(sub_months) - window_len + 1, step):
        test_start = sub_months[start]
        test_end = sub_months[start + window_len - 1]

        train_df = sub[sub["date"] < test_start].copy()
        test_df = sub[(sub["date"] >= test_start) & (sub["date"] <= test_end)].copy()

        if train_df.empty or test_df.empty:
            continue

        wname = f"{commodity_name}:{pd.Timestamp(test_start).date()}__{pd.Timestamp(test_end).date()}"
        for target_variant in target_variants:
            wres = evaluate_window(
                train_df=train_df,
                test_df=test_df,
                feature_cols=feature_cols,
                target_col=target_col,
                models=models_for_rolling,
                window_name=wname,
                target_variant=target_variant,
            )
            wres["commodity"] = commodity_name
            per_commodity_results.append(wres)

if per_commodity_results:
    per_commodity_df = pd.concat(per_commodity_results, ignore_index=True)
    per_commodity_summary = (
        per_commodity_df
        .groupby(["commodity", "target_variant", "model"], as_index=False)
        .agg(
            mae_mean=("mae", "mean"),
            rmse_mean=("rmse", "mean"),
            rmse_median=("rmse", "median"),
            r2_mean=("r2", "mean"),
            r2_median=("r2", "median"),
            n_windows=("window", "nunique"),
        )
        .sort_values(["commodity", "target_variant", "rmse_mean"])
    )

    print("Commodity-wise rolling summary:")
    display(per_commodity_summary)
else:
    print("No commodity-wise rolling windows were generated.")


if per_commodity_results:
    per_commodity_summary.to_csv(OUTPUT_DIR / "model_commodity_rolling_summary.csv", index=False)


if per_commodity_results:
    # Only keep models that beat seasonal_naive by commodity/target_variant
    base = per_commodity_summary[per_commodity_summary["model"] == "seasonal_naive"][["commodity", "target_variant", "r2_mean"]].rename(columns={"r2_mean": "r2_base"})
    beat_tbl = per_commodity_summary.merge(base, on=["commodity", "target_variant"], how="left")
    beat_tbl["delta_r2_vs_seasonal_naive"] = beat_tbl["r2_mean"] - beat_tbl["r2_base"]
    beat_tbl = beat_tbl.sort_values(["commodity", "target_variant", "delta_r2_vs_seasonal_naive"], ascending=[True, True, False])
    display(beat_tbl)
    beat_tbl.to_csv(OUTPUT_DIR / "model_commodity_vs_seasonal_naive.csv", index=False)


Commodity-wise rolling summary:


,commodity,target_variant,model,mae_mean,rmse_mean,rmse_median,r2_mean,r2_median,n_windows
3,corn,deseason_winsor,seasonal_naive,0.036069,0.047730,0.045404,0.037640,0.095464,14
4,corn,deseason_winsor,xgboost,0.036220,0.047989,0.048248,0.010591,0.048962,14
1,corn,deseason_winsor,hist_gbdt,0.036986,0.048846,0.051096,-0.029065,-0.002652,14
2,corn,deseason_winsor,lightgbm,0.038171,0.049656,0.052271,-0.085190,-0.101139,14
0,corn,deseason_winsor,elastic_net,0.041100,0.051978,0.049361,-0.274308,0.064067,14
9,corn,raw,xgboost,0.035411,0.047238,0.046770,0.032907,0.083120,14
8,corn,raw,seasonal_naive,0.036069,0.047730,0.045404,0.037640,0.095464,14
6,corn,raw,hist_gbdt,0.036848,0.048129,0.049094,0.007104,0.032674,14
7,corn,raw,lightgbm,0.037002,0.048341,0.047702,-0.016327,-0.017954,14
5,corn,raw,elastic_net,0.041503,0.052284,0.048623,-0.312916,0.019471,14


,commodity,target_variant,model,mae_mean,rmse_mean,rmse_median,r2_mean,r2_median,n_windows,r2_base,delta_r2_vs_seasonal_naive
0,corn,deseason_winsor,seasonal_naive,0.036069,0.047730,0.045404,0.037640,0.095464,14,0.037640,0.000000
1,corn,deseason_winsor,xgboost,0.036220,0.047989,0.048248,0.010591,0.048962,14,0.037640,-0.027048
2,corn,deseason_winsor,hist_gbdt,0.036986,0.048846,0.051096,-0.029065,-0.002652,14,0.037640,-0.066704
3,corn,deseason_winsor,lightgbm,0.038171,0.049656,0.052271,-0.085190,-0.101139,14,0.037640,-0.122829
4,corn,deseason_winsor,elastic_net,0.041100,0.051978,0.049361,-0.274308,0.064067,14,0.037640,-0.311947
6,corn,raw,seasonal_naive,0.036069,0.047730,0.045404,0.037640,0.095464,14,0.037640,0.000000
5,corn,raw,xgboost,0.035411,0.047238,0.046770,0.032907,0.083120,14,0.037640,-0.004733
7,corn,raw,hist_gbdt,0.036848,0.048129,0.049094,0.007104,0.032674,14,0.037640,-0.030536
8,corn,raw,lightgbm,0.037002,0.048341,0.047702,-0.016327,-0.017954,14,0.037640,-0.053967
9,corn,raw,elastic_net,0.041503,0.052284,0.048623,-0.312916,0.019471,14,0.037640,-0.350556


In [8]:
# SARIMAX commodity-wise baseline + unified "beat seasonal_naive" table
sarimax_features = [
    "futures_return",
    "usd_index",
    "interest_rate",
    "vix",
    "enso_oni",
    "temperature_anomaly",
    "precipitation_anomaly",
    "drought_index",
    "disaster_event_count",
]
sarimax_features = [c for c in sarimax_features if c in panel.columns]

def _seasonal_naive_single(train_df, test_df, target_col):
    seasonal = train_df.groupby("month_num")[target_col].mean().to_dict()
    g = float(train_df[target_col].mean())
    return test_df["month_num"].map(lambda m: seasonal.get(m, g)).to_numpy()

def _fit_predict_sarimax(train_df, test_df, target_col, exog_cols):
    idx_train = pd.DatetimeIndex(train_df["date"])
    idx_test = pd.DatetimeIndex(test_df["date"])

    y_train = pd.Series(train_df[target_col].astype(float).to_numpy(), index=idx_train).asfreq("ME")
    y_test = pd.Series(test_df[target_col].astype(float).to_numpy(), index=idx_test)

    X_train = pd.DataFrame(train_df[exog_cols].astype(float).to_numpy(), index=idx_train, columns=exog_cols).asfreq("ME")
    X_train = X_train.ffill().bfill().fillna(0.0)
    X_test = pd.DataFrame(test_df[exog_cols].astype(float).to_numpy(), index=idx_test, columns=exog_cols)
    X_test = X_test.ffill().bfill().fillna(0.0)

    # Keep spec simple and robust for repeated rolling windows.
    model = SARIMAX(
        y_train,
        exog=X_train,
        order=(1, 0, 1),
        seasonal_order=(1, 0, 0, 12),
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    try:
        fit = model.fit(disp=False, maxiter=200)
        pred = np.asarray(fit.forecast(steps=len(test_df), exog=X_test), dtype=float)
    except Exception:
        # Fallback when a window is numerically unstable.
        model = SARIMAX(
            y_train,
            exog=X_train,
            order=(1, 0, 0),
            seasonal_order=(0, 0, 0, 0),
            trend="c",
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        fit = model.fit(disp=False, maxiter=150)
        pred = np.asarray(fit.forecast(steps=len(test_df), exog=X_test), dtype=float)

    return y_test.to_numpy(), pred

# Holdout + rolling, commodity-wise SARIMAX
sarimax_holdout_rows = []
sarimax_rolling_rows = []

for commodity_name, sub0 in panel.groupby("commodity"):
    sub = sub0.sort_values("date").copy()
    sub["month_num"] = sub["date"].dt.month
    sub["target_next_return"] = sub["futures_return"].shift(-1)
    sub = sub.dropna(subset=["target_next_return"]).copy()

    sub_months = np.sort(sub["date"].unique())
    if len(sub_months) < 60:
        continue

    # holdout
    holdout_cut = sub_months[-24]
    tr = sub[sub["date"] < holdout_cut].copy()
    te = sub[sub["date"] >= holdout_cut].copy()
    if len(tr) > 36 and len(te) > 0:
        y_true, pred = _fit_predict_sarimax(tr, te, "target_next_return", sarimax_features)
        s_pred = _seasonal_naive_single(tr, te, "target_next_return")
        for mname, m_pred in [("sarimax", pred), ("seasonal_naive", s_pred)]:
            sarimax_holdout_rows.append({
                "commodity": commodity_name,
                "model": mname,
                "mae": mean_absolute_error(y_true, m_pred),
                "rmse": mean_squared_error(y_true, m_pred) ** 0.5,
                "r2": r2_score(y_true, m_pred),
                "direction_acc": (np.sign(m_pred) == np.sign(y_true)).mean(),
                "n_test": len(te),
            })

    # rolling
    for start in range(start_test_idx, len(sub_months) - window_len + 1, step):
        test_start = sub_months[start]
        test_end = sub_months[start + window_len - 1]

        tr = sub[sub["date"] < test_start].copy()
        te = sub[(sub["date"] >= test_start) & (sub["date"] <= test_end)].copy()
        if len(tr) <= 36 or len(te) == 0:
            continue

        y_true, pred = _fit_predict_sarimax(tr, te, "target_next_return", sarimax_features)
        s_pred = _seasonal_naive_single(tr, te, "target_next_return")

        for mname, m_pred in [("sarimax", pred), ("seasonal_naive", s_pred)]:
            sarimax_rolling_rows.append({
                "commodity": commodity_name,
                "window": f"{commodity_name}:{pd.Timestamp(test_start).date()}__{pd.Timestamp(test_end).date()}",
                "model": mname,
                "mae": mean_absolute_error(y_true, m_pred),
                "rmse": mean_squared_error(y_true, m_pred) ** 0.5,
                "r2": r2_score(y_true, m_pred),
                "direction_acc": (np.sign(m_pred) == np.sign(y_true)).mean(),
            })

sarimax_holdout_df = pd.DataFrame(sarimax_holdout_rows)
sarimax_rolling_df = pd.DataFrame(sarimax_rolling_rows)

print("SARIMAX holdout by commodity:")
if not sarimax_holdout_df.empty:
    display(sarimax_holdout_df.sort_values(["commodity", "rmse"]))
else:
    print("No SARIMAX holdout rows.")

if not sarimax_rolling_df.empty:
    sarimax_rolling_summary = (
        sarimax_rolling_df
        .groupby(["commodity", "model"], as_index=False)
        .agg(
            mae_mean=("mae", "mean"),
            rmse_mean=("rmse", "mean"),
            r2_mean=("r2", "mean"),
            direction_acc_mean=("direction_acc", "mean"),
            n_windows=("window", "nunique"),
        )
        .sort_values(["commodity", "rmse_mean"])
    )
    print("SARIMAX rolling summary:")
    display(sarimax_rolling_summary)
else:
    sarimax_rolling_summary = pd.DataFrame()
    print("No SARIMAX rolling rows.")

# Unified "beat seasonal_naive" table from sklearn rolling summary
if "rolling_summary" in locals() and not rolling_summary.empty:
    win_tbl = rolling_summary.copy()
    out = []
    for tv in win_tbl["target_variant"].unique():
        block = win_tbl[win_tbl["target_variant"] == tv].copy()
        base = block.loc[block["model"] == "seasonal_naive", "r2_mean"]
        if len(base) == 0:
            continue
        b = float(base.iloc[0])
        block["delta_r2_vs_seasonal_naive"] = block["r2_mean"] - b
        out.append(block)
    beat_seasonal_global = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
else:
    beat_seasonal_global = pd.DataFrame()

if not beat_seasonal_global.empty:
    print("Models vs seasonal_naive (global rolling):")
    display(beat_seasonal_global.sort_values(["target_variant", "delta_r2_vs_seasonal_naive"], ascending=[True, False]))

# Save new result tables
if not sarimax_holdout_df.empty:
    sarimax_holdout_df.to_csv(OUTPUT_DIR / "sarimax_holdout_results.csv", index=False)
if not sarimax_rolling_summary.empty:
    sarimax_rolling_summary.to_csv(OUTPUT_DIR / "sarimax_rolling_summary.csv", index=False)
if not beat_seasonal_global.empty:
    beat_seasonal_global.to_csv(OUTPUT_DIR / "model_vs_seasonal_naive.csv", index=False)

SARIMAX holdout by commodity:


,commodity,model,mae,rmse,r2,direction_acc,n_test
1,corn,seasonal_naive,0.021421,0.028126,0.405029,0.666667,24
0,corn,sarimax,0.026406,0.033405,0.160753,0.666667,24
3,soybean,seasonal_naive,0.019746,0.025011,0.273642,0.583333,24
2,soybean,sarimax,0.019264,0.027823,0.101155,0.625000,24
5,wheat,seasonal_naive,0.021762,0.029328,0.178446,0.708333,24
4,wheat,sarimax,0.024633,0.030435,0.115235,0.750000,24


SARIMAX rolling summary:


,commodity,model,mae_mean,rmse_mean,r2_mean,direction_acc_mean,n_windows
0,corn,sarimax,0.035212,0.045817,0.104087,0.601190,14
1,corn,seasonal_naive,0.036069,0.047730,0.037640,0.586310,14
2,soybean,sarimax,0.032097,0.042190,0.022181,0.589286,14
3,soybean,seasonal_naive,0.032644,0.042767,-0.015426,0.598214,14
4,wheat,sarimax,0.034230,0.044673,0.088177,0.669643,14
5,wheat,seasonal_naive,0.035699,0.046597,0.016754,0.586310,14


Models vs seasonal_naive (global rolling):


,target_variant,model,mae_mean,rmse_mean,rmse_median,r2_mean,r2_median,direction_acc_mean,n_windows,delta_r2_vs_seasonal_naive
0,deseason_winsor,xgboost,0.034445,0.045662,0.045677,0.059071,0.031686,0.608135,14,0.028887
2,deseason_winsor,hist_gbdt,0.034552,0.045873,0.045955,0.055454,0.071793,0.611111,14,0.025270
1,deseason_winsor,elastic_net,0.034718,0.045806,0.045581,0.051361,0.110825,0.615079,14,0.021177
4,deseason_winsor,seasonal_naive,0.034804,0.046604,0.048286,0.030184,0.025183,0.590278,14,0.000000
3,deseason_winsor,lightgbm,0.035398,0.046561,0.044513,0.021854,-0.037497,0.606151,14,-0.008329
5,raw,xgboost,0.034130,0.044830,0.044717,0.093437,0.127846,0.625000,14,0.063254
7,raw,hist_gbdt,0.034399,0.045451,0.045024,0.069348,0.101421,0.614087,14,0.039165
6,raw,lightgbm,0.034495,0.045322,0.044251,0.067947,0.038983,0.616071,14,0.037764
8,raw,seasonal_naive,0.034804,0.046604,0.048286,0.030184,0.025183,0.590278,14,0.000000
9,raw,elastic_net,0.035452,0.046676,0.045633,0.018216,0.082083,0.591270,14,-0.011968


## Interpretation checklist
- If `r2` is near 0 or negative, current features provide weak predictive power for monthly return.
- Disaster features may still help as interaction terms (e.g., with commodity and season).
- Next iteration: rolling-window validation, add lagged macro/disaster variables, and compare classification of return sign (+/-).